<a href="https://colab.research.google.com/github/vnishchay/100-days-AI/blob/main/RAG_Implementation_Scraping_website_data_day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Async RAG System Implementation

## Contents
* Data Flow (Async Scraping, Parsing, Chunking, Embeddings, Indexing)
* Query Flow (Async Query, Retrieval, LLM Answer Generation, Metrics)
* Modularity: Code can be ported as `ingest.py` and `query.py`

---

## Set Up Imports & Globals

In [ ]:
!pip install faiss-cpu sentence-transformers rich openai numpy tqdm aiohttp beautifulsoup4

In [ ]:
import asyncio
import aiohttp
from bs4 import BeautifulSoup
import time, os, json
from tqdm import tqdm
from typing import List, Dict
import pickle
# pip install sentence-transformers faiss-cpu rich
from sentence_transformers import SentenceTransformer
import faiss
from rich import print

## Define Async Scraper — Extract Product/Solution Pages
*Uses aiohttp (async multi-page scraping), BeautifulSoup...*

We scrape all internal `/products/` and `/solutions/` links from the homepage and recursively.

In [ ]:
BASE_URL = ""  # Replace with actual

async def fetch(session, url):
    try:
        async with session.get(url, timeout=20) as resp:
            text = await resp.text()
            return url, text, None
    except Exception as e:
        return url, None, str(e)

async def extract_links(html, prefix="/products/"):
    soup = BeautifulSoup(html, 'html.parser')
    links = set()
    for tag in soup.find_all("a", href=True):
        href = tag['href']
        if href.startswith(prefix) and href.count('/') <= 3:
            links.add(BASE_URL + href if href.startswith("/") else href)
    return links

async def scrape_all(start_urls: List[str]):
    async with aiohttp.ClientSession() as session:
        urls = set(start_urls)
        all_results, errors = [], []
        for url in list(urls):
            _, html, err = await fetch(session, url)
            if not html:
                errors.append((url, err))
                continue
            # add subpage links
            for prefix in ["/products/", "/solutions/"]:
                more = await extract_links(html, prefix)
                print(more)
                urls.update(more)

        # fetch all (concurrently)
        tasks = [fetch(session, u) for u in urls]
        pages = await asyncio.gather(*tasks)
        for url, html, err in pages:
            if html:
                all_results.append((url, html))
            else:
                errors.append((url, err))
    return all_results, errors


## Data Cleaning & Parsing Helpers
*Extract title, descriptions, clean text.*

In [ ]:
def parse_page(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, 'html.parser')
    title = soup.title.text.strip() if soup.title else ""
    p_tags = soup.find_all('p')
    long_description = " ".join(p.get_text().strip() for p in p_tags)
    short_description = long_description[:256]
    return {
        "title": title,
        "url": url,
        "short_description": short_description,
        "long_description": long_description,
    }

## Chunking for Embeddings/Search
*Simple splitter by paragraphs (can later do semantic chunking)*

In [ ]:
def chunk_text(long_text: str, chunk_size=256):
    sentences = long_text.split('. ')
    chunks = []
    curr = ''
    for s in sentences:
        if len(curr) + len(s) < chunk_size:
            curr += s + '. '
        else:
            if curr:
                chunks.append(curr.strip())
            curr = s + '. '
    if curr: chunks.append(curr.strip())
    return chunks

## Embedding & Index Helpers
*Async embedding (batched), FAISS vector index*

In [ ]:
class AsyncEmbedder:
    def __init__(self, model='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model)
    async def embed_batch(self, texts: List[str], batch_size=8):
        loop = asyncio.get_event_loop()
        embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            emb = await loop.run_in_executor(None, self.model.encode, batch)
            embeddings.extend(emb)
        return embeddings

## End-to-End Ingestion Flow (Async)


In [ ]:
async def ingest_pipeline(start_urls):
    t0 = time.time()
    print('[bold green]Starting async page scrape...')
    pages, errors = await scrape_all(start_urls)
    print(f'Scraped {len(pages)} pages ({len(errors)} failed)')
    raw_dir = './data/raw/'; os.makedirs(raw_dir, exist_ok=True)
    clean_dir = './data/cleaned/'; os.makedirs(clean_dir, exist_ok=True)
    parsed, all_chunks, mapping = [], [], []
    for url, html in tqdm(pages):
        with open(f'{raw_dir}/{url.split("/")[-1]}.html', 'w', encoding='utf-8') as f:
            f.write(html)
        data = parse_page(html, url)
        with open(f'{clean_dir}/{url.split("/")[-1]}.json', 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2)
        parsed.append(data)
        # chunking
        chunks = chunk_text(data['long_description'])
        all_chunks.extend(chunks)
        mapping.extend([(url, c) for c in chunks])
    print(f'Total Chunks: {len(all_chunks)}')

    print('[bold magenta]Generating embeddings...')
    embedder = AsyncEmbedder()
    embeddings = await embedder.embed_batch(all_chunks)

    # index
    import numpy as np
    print('[bold cyan]Building FAISS index...')
    index = faiss.IndexFlatL2(len(embeddings[0]))
    index.add(np.array(embeddings).astype('float32'))
    # store mapping
    index_data = {
        'mapping': mapping,
        'index': index
    }
    # Ensure the models directory exists
    models_dir = './models/'
    os.makedirs(models_dir, exist_ok=True)
    with open(f'{models_dir}/rag_index.pkl', 'wb') as f:
        pickle.dump(index_data, f)

    # metrics
    total_time = time.time() - t0
    print(f"[bold yellow]=== Ingestion Metrics ===")
    print(f"Total Time: {total_time:.2f}s")
    print(f"Pages Scraped: {len(pages)}")
    print(f"Pages Failed: {len(errors)}")
    print(f"Total Chunks Created: {len(all_chunks)}")
    print(f"Errors: {errors}")
    return index_data

***Call this to run data ingestion:***

In [ ]:
await ingest_pipeline([BASE_URL + '/products', BASE_URL + '/solutions'])

Starting async page scrape...

{
    'https://www.transfi.com/products/bizpay',
    'https://www.transfi.com/products/checkouts',
    'https://www.transfi.com/products/ramp',
    'https://www.transfi.com/products/wallet',
    'https://www.transfi.com/products/payouts',
    'https://www.transfi.com/products/single-api',
    'https://www.transfi.com/products/token-listing',
    'https://www.transfi.com/products/collections',
    'https://www.transfi.com/products/ramp-widget'
}

{
    'https://www.transfi.com/solutions/gaming-businesses-payment',
    'https://www.transfi.com/solutions/dollar-based-apps-payment',
    'https://www.transfi.com/solutions/startups',
    'https://www.transfi.com/solutions/web3-native-businesses-payment',
    'https://www.transfi.com/solutions/enterprises',
    'https://www.transfi.com/solutions/payroll',
    'https://www.transfi.com/solutions/payment-service-provider',
    'https://www.transfi.com/solutions/payment-gateway'
}

{
    'https://www.transfi.com/products/bizpay',
    'https://www.transfi.com/products/checkouts',
    'https://www.transfi.com/products/ramp',
    'https://www.transfi.com/products/wallet',
    'https://www.transfi.com/products/payouts',
    'https://www.transfi.com/products/single-api',
    'https://www.transfi.com/products/token-listing',
    'https://www.transfi.com/products/collections',
    'https://www.transfi.com/products/ramp-widget'
}

{
    'https://www.transfi.com/solutions/gaming-businesses-payment',
    'https://www.transfi.com/solutions/dollar-based-apps-payment',
    'https://www.transfi.com/solutions/startups',
    'https://www.transfi.com/solutions/web3-native-businesses-payment',
    'https://www.transfi.com/solutions/enterprises',
    'https://www.transfi.com/solutions/payroll',
    'https://www.transfi.com/solutions/payment-service-provider',
    'https://www.transfi.com/solutions/payment-gateway'
}

...

Pages Scraped: 19

Pages Failed: 0

Total Chunks Created: 1807

Errors: []

{'mapping': [('https://www.transfi.com/products/bizpay',
   'Unlock the world of borderless payments and experience innovative ways to move money globally Pay your employees, freelancers, vendors, suppliers and trade partners across the world with ease and at low costs Collect payments from customers around the world using payment links, with real-time settlement, easy onboarding, and low costs.'),
  ('https://www.transfi.com/products/bizpay',
   'Buy & sell any digital asset. Easy, fast & affordable. 200+ global payment methods. Supporting businesses and individuals with efficient payment solutions, helping them achieve economic prosperity through borderless finance and fostering growth globally.'),
  ('https://www.transfi.com/products/bizpay',
   'Bank, Wallet, & EMIs \xa0- Check our Digital Payment Facilities Payment Service Providers, Payment Processors, Payment Gateways & Platforms Get in touch if you offer remittance, salary, or payroll solutions! Want to enable pay-ins to dollar

# Query Flow (Async) — Batch Self-contained RAG QA

For demo, we use OpenAI LLM (or local if available).

In [ ]:
def answer_query(query, rag_index_path='./models/rag_index.pkl'):
    with open(rag_index_path, 'rb') as f:
        data = pickle.load(f)
    mapping, index = data['mapping'], data['index']
    embedder = AsyncEmbedder()

    # context = List[(url, snippet)]
    context = search_chunks(query, index, mapping, lambda x: embedder.model.encode(x))

    # Prompt includes formatting of numbered snippets
    prompt = (
        "You are a TransFi product expert. Cite sources inline as [n]. Use only information from below and list sources at the end.\n\n"
        f"Context:\n\n{format_context_for_prompt(context)}\n\nQ: {query}"
    )

    t0 = time.time()
    try:
        gemini_model = genai.GenerativeModel('gemini-2.5-flash')
        response = gemini_model.generate_content([prompt])
        gpt_answer = response.text
    except Exception as e:
        gpt_answer = f"Error generating response with Gemini: {e}"

    latency = time.time() - t0
    print(f'Question: {query}\nAnswer: {gpt_answer}')
    print('Sources:')
    for idx, (url, snippet) in enumerate(context, 1):
        print(f'[{idx}] {url}\n    "{snippet[:128]}..."')
    print(f'Metrics: Total Latency: {latency:.2f}s | Top Docs: {len(context)}')


In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Ensure the API key is configured
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
except userdata.SecretNotFoundError:
    print("GOOGLE_API_KEY not found in Colab secrets. Please add it to list models.")
except Exception as e:
    print(f"An error occurred while setting up the Gemini API: {e}")


print("Available models that support generateContent:")
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

Available models that support generateContent:

models/gemini-2.5-pro-preview-03-25

models/gemini-2.5-flash-preview-05-20

models/gemini-2.5-flash

models/gemini-2.5-flash-lite-preview-06-17

...

models/gemini-2.5-flash-image

models/gemini-2.5-flash-preview-09-2025

models/gemini-2.5-flash-lite-preview-09-2025

models/gemini-robotics-er-1.5-preview

models/gemini-2.5-computer-use-preview-10-2025

In [ ]:
answer_query("What is one unique feature of TransFi's product? tell me in one line")

Question: What is one unique feature of TransFi's product? tell me in one line
Answer: As a TransFi product expert:

One unique feature of TransFi's product is its **optimized global fees coupled with complete transparency**, 
ensuring cost-effective transactions for both users and businesses.

**Source:**
"With optimized global fees and complete transparency, TransFi ensures every transaction remains fast, seamless, 
and cost-effective for both users and businesses."
*(Source: Provided Context)*

Sources:

- https://www.transfi.com/solutions/startups: "From your inaugural transaction to millions in revenue, TransFi 
adapts with meticulous precision, providing infrastructure that ..."

- https://www.transfi.com/solutions/payment-gateway: "“ I’ve had the pleasure of working with TransFi for the past 
year, and their commitment and support have been exemplary. The tea..."

- https://www.transfi.com/products/single-api: "TransFi lets you seamlessly integrate digital assets into your 
platform, enabling your users to buy, sell, and transfer stableco..."

- https://www.transfi.com/products/token-listing: "With optimized global fees and complete transparency, TransFi 
ensures every transaction remains fast, seamless, and cost-effecti..."

- https://www.transfi.com/solutions/startups: "Operate with perspicacious assurance: TransFi ensures full adherence
to global financial regulations, delivering secure, auditab..."

Metrics: Total Latency: 10.42s | Top Docs: 5